In [1]:
!pip install -q transformers sentencepiece

In [2]:
!pip install -q evaluate sacrebleu

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import evaluate

In [4]:
model_name = 'Helsinki-NLP/opus-mt-en-uk'

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.01M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.37M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  305MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [6]:
text = ["Is overtourism pricing locals out?",
        "If you have ever tried to enjoy a picturesque old town, only to find it full of souvenir shops, short-term rentals, and restaurants with menus in five languages, you have most likely experienced overtourism.",
        "If you have ever seen graffiti reading ‘Tourists go home’, you have definitely seen how serious the issue has become.",
        "In recent years, travelling has become easier than ever.",
        "Rarely have so many people travelled so often, and as more flats are turned into holiday rentals, prices go up, making it harder and harder for locals to live in the city centre.",
        "‘Where are all the Amsterdamers?’ I asked an underground worker during a visit to Amsterdam, a city I had lived in 20 years earlier as an exchange student, but which I now barely recognised.",
        "‘We can’t afford to live here anymore,’ he replied."
]

In [7]:
batch = tokenizer(text, return_tensors='pt', padding=True)
generated_tokens = model.generate(**batch, max_new_tokens=150)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [8]:
translated_sentences = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
for i in range(len(text)):
  print(translated_sentences[i])

Чи місцевою мовою є надмірна плата за ціноутворення?
Якщо ви коли-небудь намагалися насолодитися мальовничим старим містом, тільки щоб знайти його повним магазинів сувенірів, короткострокових оренд та ресторанів з меню на п'ять мов, ви, ймовірно, зазнали надмірного захоплення.
Якщо ви колись бачили, як графіті читається вдома, то ви чітко бачите, наскільки серйозною стала проблема.
В останні роки подорожі стали, як ніколи, простішими.
Рідко коли так багато людей подорожували так часто, і коли більше квартир перетворюються на святкові оренди, ціни зростають, що ускладнює і ускладнює життя місцевих жителів у центрі міста.
Я попросив підпільного працівника під час візиту в Амстердам, місто, яке я жив 20 років тому як студент обміну, але якого я зараз ледве впізнав.
Ми не можемо дозволити собі жити тут, він відповів.


In [9]:
bleu_metric = evaluate.load("sacrebleu")
references = [
    ["Чи витісняє надмірний туризм місцевих жителів через зростання цін?"],
    ["Якщо ви коли-небудь намагалися насолодитися мальовничим старим містом, а натомість знаходили його переповненим сувенірними крамницями, житлом для короткострокової оренди та ресторанами з меню п'ятьма мовами, ви, швидше за все, стикалися з надмірним туризмом."],
    ["Якщо ви коли-небудь бачили графіті з написом «Туристи, їдьте додому», ви точно знаєте, наскільки серйозною стала ця проблема."],
    ["Останніми роками подорожувати стало простіше, ніж будь-коли."],
    ["Рідко коли так багато людей подорожували так часто, і оскільки все більше квартир перетворюються на житло для відпочинку, ціни зростають, через що місцевим жителям стає дедалі важче жити в центрі міста."],
    ["«Де всі амстердамці?» — запитав я працівника метро під час візиту до Амстердама, міста, в якому я жив 20 років тому як студент за обміном, але яке зараз ледве впізнав."],
    ["«Ми більше не можемо дозволити собі тут жити», — відповів він."]
]

In [12]:
results = bleu_metric.compute(predictions=translated_sentences, references=references)
print(results)

{'score': 27.431562464459827, 'counts': [89, 48, 33, 22], 'totals': [147, 140, 133, 126], 'precisions': [60.54421768707483, 34.285714285714285, 24.81203007518797, 17.46031746031746], 'bp': 0.8907902770561547, 'sys_len': 147, 'ref_len': 164}


In general, the result is not bad, but could be significantly improved